# Baseline prediction-model comparison

This notebook builds the first predictive modelling workflow using the cleaned diabetes dataset. It prepares the model features, applies a common preprocessing pipeline, trains several baseline models, and compares their predictive performance.


## 1. Data preparation

This section loads the cleaned dataset, defines the binary 30-day readmission target, selects the modelling features, creates the train/test split, and builds the preprocessing pipeline shared by the models.


In [49]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


### Locate and load the cleaned dataset

The code searches upward from the current working directory until it finds `Processed_Dataset/diabetic_data_cleaned_stage1.csv`. This makes the notebook less dependent on being launched from one exact folder.


In [50]:
PROJECT_ROOT = Path.cwd()

# Search the current directory and its parents so the notebook still works
# when it is opened from a subfolder of the project.
for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv")

df = pd.read_csv(DATA_PATH)

print(DATA_PATH)
print(df.shape)
df.head()

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv
(69987, 56)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,diabetesMed,readmitted,readmitted_30,hba1c_group,primary_diagnosis,age_group,discharge_group,race_group,admission_source_group,medical_specialty_group
0,24437208,135,Caucasian,Female,[50-60),2,1,1,8,Cardiology,...,Yes,<30,1,No test was performed,Circulatory,30-60,Home,Caucasian,Physician/clinic referral,Cardiology
1,29758806,378,Caucasian,Female,[50-60),3,1,1,2,Surgery-Neuro,...,No,NO,0,No test was performed,Musculoskeletal,30-60,Home,Caucasian,Physician/clinic referral,Surgery
2,189899286,729,Caucasian,Female,[80-90),1,3,7,4,InternalMedicine,...,Yes,NO,0,Normal result of the test,Injury,>60,Other,Caucasian,Emergency room,Internal Medicine
3,64331490,774,Caucasian,Female,[80-90),1,1,7,3,InternalMedicine,...,Yes,NO,0,"High, medication changed",Other,>60,Home,Caucasian,Emergency room,Internal Medicine
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,7,5,InternalMedicine,...,Yes,NO,0,No test was performed,Genitourinary,30-60,Home,AfricanAmerican,Emergency room,Internal Medicine


In [51]:
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted',
 'readmitted_30',
 'hba1c_group',
 'primary_diagnosis',
 'age_group',
 'discharge_group',
 'race_group',
 'admission_source_group',
 'medical_specialty_group']

In [52]:
df["readmitted"].value_counts(dropna=False)

readmitted
NO     41476
>30    22226
<30     6285
Name: count, dtype: int64

In [53]:
# Binary target: 1 = readmitted within 30 days, 0 = all other outcomes.
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

df["readmitted_30"].value_counts(normalize=True)

readmitted_30
0    0.910198
1    0.089802
Name: proportion, dtype: float64

### Select model inputs

The model uses the grouped variables created during preprocessing plus several numerical hospital-utilisation variables. Raw IDs, identifiers, the original readmission label, and raw versions of already-grouped variables are deliberately excluded.


In [ ]:
target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed"
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# Keep one master list so the exact model input set is explicit and reproducible.
model_features = categorical_features + numeric_features

# Fail early if the preprocessing notebook did not create an expected feature.
missing_features = [
    col for col in model_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"These modelling features are missing: {missing_features}"
    )

X = df[model_features].copy()
y = df[target_col].astype(int).copy()


print("X shape:", X.shape)
print("Features used:", X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

# Sanity check: raw/identifier/target columns must not leak into X.
forbidden_features = {
    "race",
    "age",
    "medical_specialty",
    "diag_1",
    "A1Cresult",
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id",
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr"
}

unexpected_features = forbidden_features.intersection(X.columns)

assert not unexpected_features, (
    f"Unexpected features found in X: {unexpected_features}"
)

assert not X.columns.duplicated().any(), (
    "Duplicate column names found in X."
)

print("Feature-selection checks passed.")

X shape: (69987, 18)
Features used: ['gender', 'race_group', 'age_group', 'admission_source_group', 'discharge_group', 'medical_specialty_group', 'primary_diagnosis', 'hba1c_group', 'max_glu_serum', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Target distribution:
readmitted_30
0    63702
1     6285
Name: count, dtype: int64
Feature-selection checks passed.


### Create the initial train/test split

The split is stratified so the rare 30-day readmission class keeps approximately the same proportion in both sets. The fixed random seed makes the split reproducible.


In [55]:
# stratify=y preserves the readmission prevalence in both partitions.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, y_train.mean())
print("Test:", X_test.shape, y_test.mean())

Train: (55989, 18) 0.08980335423029524
Test: (13998, 18) 0.08979854264894985


### Shared preprocessing pipeline

Categorical and numerical columns need different preprocessing. Categorical values are imputed and one-hot encoded; numerical values are median-imputed and standardised. `ColumnTransformer` applies both branches in a single reusable preprocessing step.


In [56]:
# Categorical branch: fill missing values with the most frequent category,
# then one-hot encode categories; unseen test categories are ignored safely.
# categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
# numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

# print("Categorical features:", len(categorical_features))
# print(categorical_features)

# print("Numeric features:", len(numeric_features))
# print(numeric_features)

categorical_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# Numerical branch: median imputation followed by standardisation.
numeric_transformer = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

# Combine both branches so every downstream model receives the same transformed data.
preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ],
    remainder="drop"
)

### Shared preprocessing pipeline

Categorical and numerical columns need different preprocessing. Categorical values are imputed and one-hot encoded; numerical values are median-imputed and standardised. `ColumnTransformer` applies both branches in a single reusable preprocessing step.


In [57]:
# Categorical branch: fill missing values with the most frequent category,
# then one-hot encode categories; unseen test categories are ignored safely.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Numerical branch: median imputation followed by standardisation.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Combine both branches so every downstream model receives the same transformed data.
preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features)
    ]
)

### Evaluation helper: confusion-matrix output

This helper generates both a conventional 2×2 confusion matrix and a one-row summary containing TP, TN, FP, FN and the corresponding rates. The same function is reused for every model so the saved outputs are consistent.


In [58]:
def save_confusion_matrix(model, X_test, y_test, model_name, output_dir):
    """Create and save confusion-matrix outputs for one fitted model.

    Returns both:
    1. a 2×2 matrix for easy inspection; and
    2. a one-row table containing TN, FP, FN, TP and derived error/recall rates.
    """
    y_pred = model.predict(X_test)

    # sklearn orders a binary confusion matrix as [[TN, FP], [FN, TP]].
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual not readmitted", "Actual readmitted"],
        columns=["Predicted not readmitted", "Predicted readmitted"]
    )

    file_name = (
        model_name
        .lower()
        .replace(":", "")
        .replace(" ", "_")
        .replace("/", "_")
    )

    cm_df.to_csv(output_dir / f"confusion_matrix_{file_name}.csv")

    # Flatten the matrix into named counts for later comparison between models.
    tn, fp, fn, tp = cm.ravel()

    total = tn + fp + fn + tp

    cm_long = pd.DataFrame([{
        "model": model_name,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp,
        "total": total,
        "true_negative_rate": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "false_positive_rate": fp / (tn + fp) if (tn + fp) > 0 else 0,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) > 0 else 0,
        "true_positive_rate_recall": tp / (fn + tp) if (fn + tp) > 0 else 0
    }])

    return cm_df, cm_long

## 2. Dummy baseline

The dummy classifier always predicts the most frequent class. It provides a minimum baseline so later models can be checked against a classifier that does not learn any useful clinical pattern.


In [59]:
# The same preprocessing is included inside every model Pipeline.
# This ensures transformations are learnt from training data together with the classifier.
dummy_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

### Evaluation helper: common performance metrics

Every model is evaluated with the same set of metrics. Probability-based metrics use `predict_proba`, while the standard classification metrics use the model's class prediction.


In [60]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """Evaluate a fitted binary classifier with the same metrics used across models.

    `y_pred` is used for threshold-dependent metrics such as precision and recall.
    `y_proba` is used for ranking/probability metrics such as AUROC, AUPRC and Brier score.
    """
    y_pred = model.predict(X_test)
    
    # Prefer predicted probabilities because AUROC/AUPRC/Brier need a continuous score.
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    
    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auroc": roc_auc_score(y_test, y_proba),
        "auprc": average_precision_score(y_test, y_proba),
        "brier_score": brier_score_loss(y_test, y_proba)
    }
    
    return results

In [61]:
dummy_results = evaluate_model(
    dummy_model,
    X_test,
    y_test,
    model_name="Dummy: most frequent"
)

dummy_results

{'model': 'Dummy: most frequent',
 'accuracy': 0.9102014573510502,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'auroc': 0.5,
 'auprc': 0.08979854264894985,
 'brier_score': 0.08979854264894985}

In [62]:
results_df = pd.DataFrame([dummy_results])
results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.0,0.0,0.0,0.5,0.089799,0.089799


In [63]:
OUTPUT_DIR = PROJECT_ROOT / "Model_Results"
OUTPUT_DIR.mkdir(exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "model_comparison_initial.csv", index=False)
dummy_cm_df, dummy_cm_long = save_confusion_matrix(
    model=dummy_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Dummy: most frequent",
    output_dir=OUTPUT_DIR
)

In [64]:
dummy_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,12741,0
Actual readmitted,1257,0


In [65]:
dummy_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Dummy: most frequent,12741,0,1257,0,13998,1.0,0.0,1.0,0.0


In [66]:
extra_drop_cols = [
    col for col in df.columns
    if "table" in col.lower() or "display" in col.lower()
]

extra_drop_cols

[]

In [67]:
y_dummy_pred = dummy_model.predict(X_test)

print(classification_report(
    y_test,
    y_dummy_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.91      1.00      0.95     12741
    Readmitted       0.00      0.00      0.00      1257

      accuracy                           0.91     13998
     macro avg       0.46      0.50      0.48     13998
  weighted avg       0.83      0.91      0.87     13998



## 3. Logistic Regression

This model provides a simple linear baseline. Class weighting is used because 30-day readmission is the minority class, so mistakes on readmitted patients receive more weight during training.


In [68]:
# Balanced class weights increase the loss contribution of the minority class.
log_reg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

In [69]:
log_reg_results = evaluate_model(
    log_reg_model,
    X_test,
    y_test,
    model_name="Logistic Regression"
)

log_reg_results

{'model': 'Logistic Regression',
 'accuracy': 0.6153736248035434,
 'precision': 0.12999820692128383,
 'recall': 0.5767700875099443,
 'f1': 0.21217442200760903,
 'auroc': 0.635112173336263,
 'auprc': 0.14854107870676367,
 'brier_score': 0.23594714296114733}

In [70]:
results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

log_reg_cm_df, log_reg_cm_long = save_confusion_matrix(
    model=log_reg_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Logistic Regression",
    output_dir=OUTPUT_DIR
)

In [71]:
log_reg_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,7889,4852
Actual readmitted,532,725


In [72]:
log_reg_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Logistic Regression,7889,4852,532,725,13998,0.619182,0.380818,0.42323,0.57677


In [73]:
y_logreg_pred = log_reg_model.predict(X_test)

print(classification_report(
    y_test,
    y_logreg_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.94      0.62      0.75     12741
    Readmitted       0.13      0.58      0.21      1257

      accuracy                           0.62     13998
     macro avg       0.53      0.60      0.48     13998
  weighted avg       0.86      0.62      0.70     13998



## 4. Regularised Logistic Regression

This extends the logistic-regression baseline by tuning the regularisation type and strength. Grid search uses cross-validated AUPRC (`average_precision`) to choose the best combination.


### Tune penalty type and regularisation strength

`GridSearchCV` tries every combination of L1/L2 penalty and the candidate `C` values. A smaller `C` means stronger regularisation. Three-fold cross-validation chooses the setting with the highest mean AUPRC.


In [74]:
regularised_log_reg_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=5000,
        tol=1e-3,
        random_state=42
    ))
])

# `model__...` tells GridSearchCV that these parameters belong to the
# classifier inside the Pipeline.
regularised_log_reg_param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1, 10]
}

# average_precision is the sklearn scoring name used here for AUPRC.
regularised_log_reg_search = GridSearchCV(
    estimator=regularised_log_reg_pipeline,
    param_grid=regularised_log_reg_param_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1
)

regularised_log_reg_search.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaco

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=0.001))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1', 'l2']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`

In [75]:
regularised_log_reg_search.best_params_

{'model__C': 0.01, 'model__penalty': 'l1'}

In [76]:
regularised_log_reg_search.best_score_

np.float64(0.14581811135969938)

In [77]:
# best_estimator_ is already refitted on the full training set using the
# hyperparameters that achieved the best mean cross-validation AUPRC.
regularised_log_reg_model = regularised_log_reg_search.best_estimator_

reg_log_reg_results = evaluate_model(
    regularised_log_reg_model,
    X_test,
    y_test,
    model_name="Regularised Logistic Regression"
)

reg_log_reg_results

{'model': 'Regularised Logistic Regression',
 'accuracy': 0.6185883697671096,
 'precision': 0.13304566702624954,
 'recall': 0.588703261734288,
 'f1': 0.2170406217920516,
 'auroc': 0.6381830542619599,
 'auprc': 0.14979324840545777,
 'brier_score': 0.23614176024475111}

In [78]:
# Rebuild the comparison table safely.
# This avoids duplicate rows if you rerun the notebook cells.

available_results = []

if "dummy_results" in globals():
    available_results.append(dummy_results)

if "log_reg_results" in globals():
    available_results.append(log_reg_results)

if "reg_log_reg_results" in globals():
    available_results.append(reg_log_reg_results)

results_df = pd.DataFrame(available_results)

results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.000000,0.000000,0.000000,0.500000,0.089799,0.089799
1,Logistic Regression,0.615374,0.129998,0.576770,0.212174,0.635112,0.148541,0.235947
2,Regularised Logistic Regression,0.618588,0.133046,0.588703,0.217041,0.638183,0.149793,0.236142


In [79]:
reg_log_reg_cm_df, reg_log_reg_cm_long = save_confusion_matrix(
    model=regularised_log_reg_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Regularised Logistic Regression",
    output_dir=OUTPUT_DIR
)

In [80]:
reg_log_reg_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,7919,4822
Actual readmitted,517,740


In [81]:
reg_log_reg_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Regularised Logistic Regression,7919,4822,517,740,13998,0.621537,0.378463,0.411297,0.588703


In [82]:
y_reg_logreg_pred = regularised_log_reg_model.predict(X_test)

print(classification_report(
    y_test,
    y_reg_logreg_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.94      0.62      0.75     12741
    Readmitted       0.13      0.59      0.22      1257

      accuracy                           0.62     13998
     macro avg       0.54      0.61      0.48     13998
  weighted avg       0.87      0.62      0.70     13998



In [83]:
regularised_log_reg_best_params = pd.DataFrame([{
    "model": "Regularised Logistic Regression",
    "best_penalty": regularised_log_reg_search.best_params_["model__penalty"],
    "best_C": regularised_log_reg_search.best_params_["model__C"],
    "best_cv_auprc": regularised_log_reg_search.best_score_
}])

regularised_log_reg_best_params.to_csv(
    OUTPUT_DIR / "regularised_logistic_regression_best_params.csv",
    index=False
)

regularised_log_reg_best_params

,model,best_penalty,best_C,best_cv_auprc
0,Regularised Logistic Regression,l1,0.01,0.145818


## 5. Decision Tree

A shallow decision tree is trained as a non-linear baseline. Depth and minimum leaf size are restricted to reduce overfitting, and balanced class weights are again used for the imbalanced target.


In [84]:
# max_depth and min_samples_leaf deliberately restrict tree complexity.
decision_tree_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42
    ))
])

decision_tree_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

In [85]:
decision_tree_results = evaluate_model(
    decision_tree_model,
    X_test,
    y_test,
    model_name="Decision Tree"
)

decision_tree_results

{'model': 'Decision Tree',
 'accuracy': 0.5667238176882412,
 'precision': 0.12612752721617418,
 'recall': 0.645186953062848,
 'f1': 0.21100559385976322,
 'auroc': 0.6256919495858901,
 'auprc': 0.1341446492081273,
 'brier_score': 0.23694667882312567}

In [86]:
decision_tree_cm_df, decision_tree_cm_long = save_confusion_matrix(
    model=decision_tree_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Decision Tree",
    output_dir=OUTPUT_DIR
)

In [87]:
decision_tree_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,7122,5619
Actual readmitted,446,811


In [88]:
decision_tree_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Decision Tree,7122,5619,446,811,13998,0.558983,0.441017,0.354813,0.645187


In [89]:
y_decision_tree_pred = decision_tree_model.predict(X_test)

print(classification_report(
    y_test,
    y_decision_tree_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.94      0.56      0.70     12741
    Readmitted       0.13      0.65      0.21      1257

      accuracy                           0.57     13998
     macro avg       0.53      0.60      0.46     13998
  weighted avg       0.87      0.57      0.66     13998



In [90]:
# Get AUROC score
auroc = roc_auc_score(y_test, y_decision_tree_pred)
print(f"AUROC Score: {auroc}")

AUROC Score: 0.6020848822295638


### Inspect which encoded features the tree actually used

The trained tree stores the feature index used at each internal split. These indices are mapped back to the one-hot encoded feature names so the variables used by the fitted tree can be inspected.


In [91]:
# Access the fitted classifier inside the Pipeline.
tree_classifier = decision_tree_model.named_steps["model"]

# The preprocessing step expands categorical variables into one-hot columns,
# so retrieve those transformed names before mapping tree split indices back to features.
feature_names = (
    decision_tree_model
    .named_steps["preprocess"]
    .get_feature_names_out()
)

# Internal nodes have a feature index of 0 or greater.
# Leaf nodes use a negative index.
used_feature_indices = np.unique(
    tree_classifier.tree_.feature[
        tree_classifier.tree_.feature >= 0
    ]
)

used_features = pd.DataFrame({
    "feature": feature_names[used_feature_indices],
    "importance": tree_classifier.feature_importances_[used_feature_indices]
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

used_features

,feature,importance
0,cat__discharge_group_Home,0.430739
1,num__number_inpatient,0.314238
2,num__time_in_hospital,0.068395
3,cat__primary_diagnosis_Circulatory,0.040898
4,cat__age_group_>60,0.032659
5,num__number_diagnoses,0.020843
6,cat__diabetesMed_No,0.020547
7,num__num_lab_procedures,0.019049
8,cat__primary_diagnosis_Respiratory,0.012871
9,cat__admission_source_group_Other,0.012541


In [92]:
print("Actual depth:", tree_classifier.get_depth())
print("Number of leaves:", tree_classifier.get_n_leaves())
print("Number of used encoded features:", len(used_features))

Actual depth: 5
Number of leaves: 29
Number of used encoded features: 14


## 6. Random Forest

The Random Forest combines many decision trees. A randomised hyperparameter search explores different forest sizes, depths, split constraints, leaf sizes, and feature-sampling settings using cross-validated AUPRC.


In [93]:
# balanced_subsample recalculates class weights for each bootstrap sample/tree.
random_forest_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

### Random Forest hyperparameter search

The search space controls the number and complexity of trees. `RandomizedSearchCV` samples 15 combinations rather than testing the full Cartesian product, which keeps the search computationally manageable.


In [94]:
# Candidate values control forest size, tree depth, node size and the
# number of features considered when choosing each split.
random_forest_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [8, 12, 16, 20, None],
    "model__min_samples_split": [2, 10, 30],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__max_features": ["sqrt", "log2", 0.5]
}

# Randomised search evaluates only 15 sampled combinations from the much
# larger search space, using mean 3-fold CV AUPRC to rank them.
random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=random_forest_param_grid,
    n_iter=15,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_forest_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


/opt/anaconda3/envs/indi_preprocessing/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [8, 12, ...], 'model__max_features': ['sqrt', 'log2', ...], 'model__min_samples_leaf': [1, 5, ...], 'model__min_samples_split': [2, 10, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimato

In [95]:
random_forest_search.best_params_

{'model__n_estimators': 500,
 'model__min_samples_split': 2,
 'model__min_samples_leaf': 10,
 'model__max_features': 'log2',
 'model__max_depth': 16}

In [96]:
random_forest_search.best_score_

np.float64(0.1421129314074074)

In [97]:
# Use the refitted best pipeline selected by cross-validation.
random_forest_model = random_forest_search.best_estimator_

rf_results = evaluate_model(
    random_forest_model,
    X_test,
    y_test,
    model_name="Random Forest"
)

rf_results

{'model': 'Random Forest',
 'accuracy': 0.7497499642806115,
 'precision': 0.14484503478810878,
 'recall': 0.3643595863166269,
 'f1': 0.2072867164516859,
 'auroc': 0.634998470538144,
 'auprc': 0.14097527329731985,
 'brier_score': 0.18519148131273933}

In [98]:
# Rebuild the comparison table from whichever model-result dictionaries
# have already been created. The globals checks make rerunning sections safer.
available_results = []

if "dummy_results" in globals():
    available_results.append(dummy_results)

if "log_reg_results" in globals():
    available_results.append(log_reg_results)

if "reg_log_reg_results" in globals():
    available_results.append(reg_log_reg_results)

if "rf_results" in globals():
    available_results.append(rf_results)

results_df = pd.DataFrame(available_results)

results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.000000,0.000000,0.000000,0.500000,0.089799,0.089799
1,Logistic Regression,0.615374,0.129998,0.576770,0.212174,0.635112,0.148541,0.235947
2,Regularised Logistic Regression,0.618588,0.133046,0.588703,0.217041,0.638183,0.149793,0.236142
3,Random Forest,0.749750,0.144845,0.364360,0.207287,0.634998,0.140975,0.185191


In [99]:
rf_cm_df, rf_cm_long = save_confusion_matrix(
    model=random_forest_model,
    X_test=X_test,
    y_test=y_test,
    model_name="Random Forest",
    output_dir=OUTPUT_DIR
)

In [100]:
rf_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,10037,2704
Actual readmitted,799,458


In [101]:
rf_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,Random Forest,10037,2704,799,458,13998,0.787772,0.212228,0.63564,0.36436


In [102]:
y_rf_pred = random_forest_model.predict(X_test)

print(classification_report(
    y_test,
    y_rf_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.93      0.79      0.85     12741
    Readmitted       0.14      0.36      0.21      1257

      accuracy                           0.75     13998
     macro avg       0.54      0.58      0.53     13998
  weighted avg       0.86      0.75      0.79     13998



In [103]:
random_forest_best_params = pd.DataFrame([{
    "model": "Random Forest",
    "best_n_estimators": random_forest_search.best_params_["model__n_estimators"],
    "best_max_depth": random_forest_search.best_params_["model__max_depth"],
    "best_min_samples_split": random_forest_search.best_params_["model__min_samples_split"],
    "best_min_samples_leaf": random_forest_search.best_params_["model__min_samples_leaf"],
    "best_max_features": random_forest_search.best_params_["model__max_features"],
    "best_cv_auprc": random_forest_search.best_score_
}])

random_forest_best_params.to_csv(
    OUTPUT_DIR / "random_forest_best_params.csv",
    index=False
)

random_forest_best_params

,model,best_n_estimators,best_max_depth,best_min_samples_split,best_min_samples_leaf,best_max_features,best_cv_auprc
0,Random Forest,500,16,2,10,log2,0.142113


In [104]:
# Align transformed feature names with the forest's impurity-based importance values.
rf_feature_names = random_forest_model.named_steps["preprocess"].get_feature_names_out()

rf_feature_importance = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": random_forest_model.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

rf_feature_importance.to_csv(
    OUTPUT_DIR / "random_forest_feature_importance.csv",
    index=False
)

rf_feature_importance.head(30)

,feature,importance
39,num__num_lab_procedures,0.126576
41,num__num_medications,0.108974
38,num__time_in_hospital,0.083715
45,num__number_diagnoses,0.062991
44,num__number_inpatient,0.058003
40,num__num_procedures,0.052227
12,cat__discharge_group_Home,0.047196
13,cat__discharge_group_Other,0.040933
42,num__number_outpatient,0.021463
17,cat__medical_specialty_group_Missing,0.019116


## 7. XGBoost

XGBoost provides a gradient-boosted tree baseline. The positive-class weight is calculated from the training data so the model gives more attention to the much smaller readmitted class.


### Calculate the XGBoost class-imbalance weight

`scale_pos_weight` is set to the number of negative training cases divided by the number of positive cases. This increases the contribution of the minority readmission class to the XGBoost training objective.


In [105]:
# Calculate the class-imbalance ratio from TRAINING data only.
# scale_pos_weight = number of negative cases / number of positive cases
# A value above 1 gives the minority positive/readmission class more influence.

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

scale_pos_weight

np.float64(10.1354415274463)

In [106]:
# These are baseline XGBoost settings rather than a hyperparameter search.
# Subsampling rows/features adds regularisation, while scale_pos_weight addresses imbalance.
xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

xgb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['gender','race_group','age_group',...,'number_emergency', 'number_inpatient','number_diagnoses']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

In [107]:
xgb_results = evaluate_model(
    xgb_pipeline,
    X_test,
    y_test,
    model_name="XGBoost"
)

xgb_results

{'model': 'XGBoost',
 'accuracy': 0.6178739819974282,
 'precision': 0.13516405135520684,
 'recall': 0.6030230708035004,
 'f1': 0.22083029861616899,
 'auroc': 0.6413630486636113,
 'auprc': 0.1482237645365625,
 'brier_score': 0.22986316680908203}

In [108]:
# Rebuild the comparison table from whichever model-result dictionaries
# have already been created. The globals checks make rerunning sections safer.
available_results = []

if "dummy_results" in globals():
    available_results.append(dummy_results)

if "log_reg_results" in globals():
    available_results.append(log_reg_results)

if "reg_log_reg_results" in globals():
    available_results.append(reg_log_reg_results)

if "rf_results" in globals():
    available_results.append(rf_results)

if "xgb_results" in globals():
    available_results.append(xgb_results)

results_df = pd.DataFrame(available_results)

results_df.to_csv(
    OUTPUT_DIR / "model_comparison_initial.csv",
    index=False
)

results_df

,model,accuracy,precision,recall,f1,auroc,auprc,brier_score
0,Dummy: most frequent,0.910201,0.000000,0.000000,0.000000,0.500000,0.089799,0.089799
1,Logistic Regression,0.615374,0.129998,0.576770,0.212174,0.635112,0.148541,0.235947
2,Regularised Logistic Regression,0.618588,0.133046,0.588703,0.217041,0.638183,0.149793,0.236142
3,Random Forest,0.749750,0.144845,0.364360,0.207287,0.634998,0.140975,0.185191
4,XGBoost,0.617874,0.135164,0.603023,0.220830,0.641363,0.148224,0.229863


In [109]:
xgb_cm_df, xgb_cm_long = save_confusion_matrix(
    model=xgb_pipeline,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost",
    output_dir=OUTPUT_DIR
)

In [110]:
xgb_cm_df

,Predicted not readmitted,Predicted readmitted
Actual not readmitted,7891,4850
Actual readmitted,499,758


In [111]:
xgb_cm_long

,model,true_negative,false_positive,false_negative,true_positive,total,true_negative_rate,false_positive_rate,false_negative_rate,true_positive_rate_recall
0,XGBoost,7891,4850,499,758,13998,0.619339,0.380661,0.396977,0.603023


In [112]:
y_xgb_pred = xgb_pipeline.predict(X_test)

print(classification_report(
    y_test,
    y_xgb_pred,
    target_names=["Not readmitted", "Readmitted"],
    zero_division=0
))

                precision    recall  f1-score   support

Not readmitted       0.94      0.62      0.75     12741
    Readmitted       0.14      0.60      0.22      1257

      accuracy                           0.62     13998
     macro avg       0.54      0.61      0.48     13998
  weighted avg       0.87      0.62      0.70     13998



In [113]:
xgb_baseline_params = pd.DataFrame([{
    "model": "XGBoost",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight
}])

xgb_baseline_params.to_csv(
    OUTPUT_DIR / "xgboost_baseline_params.csv",
    index=False
)

xgb_baseline_params

,model,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,scale_pos_weight
0,XGBoost,300,0.05,3,0.8,0.8,10.135442


In [114]:
xgb_baseline_params = pd.DataFrame([{
    "model": "XGBoost",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight
}])

xgb_baseline_params.to_csv(
    OUTPUT_DIR / "xgboost_baseline_params.csv",
    index=False
)

xgb_baseline_params

,model,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,scale_pos_weight
0,XGBoost,300,0.05,3,0.8,0.8,10.135442


### Extract XGBoost feature importance

The transformed feature names are aligned with the fitted XGBoost importance values, then sorted so the most influential encoded features can be inspected and saved.


In [115]:
# Match each fitted XGBoost importance value to its post-preprocessing feature name.
xgb_feature_names = xgb_pipeline.named_steps["preprocess"].get_feature_names_out()

xgb_feature_importance = pd.DataFrame({
    "feature": xgb_feature_names,
    "importance": xgb_pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

xgb_feature_importance.to_csv(
    OUTPUT_DIR / "xgboost_feature_importance_baseline.csv",
    index=False
)

xgb_feature_importance.head(30)

,feature,importance
12,cat__discharge_group_Home,0.153941
13,cat__discharge_group_Other,0.126311
44,num__number_inpatient,0.057738
8,cat__age_group_>60,0.049185
20,cat__primary_diagnosis_Circulatory,0.025875
36,cat__diabetesMed_No,0.025170
37,cat__diabetesMed_Yes,0.024027
38,num__time_in_hospital,0.020817
28,cat__primary_diagnosis_Respiratory,0.020645
43,num__number_emergency,0.018911


## 8. Baseline performance

The results produced above are saved for later comparison and further tuning. At this stage the notebook mainly establishes a common modelling and evaluation workflow for the candidate models.
